# Diabetic Retinopathy Grading with PyTorch

Attach the **APTOS-2019 dataset** in Kaggle before running this notebook.
This version uses the exact files shown in your dataset sidebar.


In [ ]:
import os
import cv2
import torch
import pandas as pd
from PIL import Image
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import EfficientNet_B3_Weights, efficientnet_b3

CANDIDATES = [
    "/kaggle/input/datasets/mariaherrerot/aptos2019",
    "/kaggle/input/aptos2019",
    "/kaggle/input/aptos-2019-dataset",
]
DATA_DIR = next((p for p in CANDIDATES if os.path.exists(os.path.join(p, "train_1.csv"))), None)
if DATA_DIR is None:
    raise FileNotFoundError("Update CANDIDATES to match your Kaggle dataset mount path.")
TRAIN_CSV = os.path.join(DATA_DIR, "train_1.csv")
VAL_CSV = os.path.join(DATA_DIR, "valid.csv")
TRAIN_DIR = os.path.join(DATA_DIR, "train_images")
VAL_DIR = os.path.join(DATA_DIR, "val_images")
MODEL_PATH = "/kaggle/working/best_model.pth"
BATCH_SIZE = 16
EPOCHS = 20
LR = 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dataset path:", DATA_DIR)
print("Using device:", device)


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

class RetinaDataset(Dataset):
    def __init__(self, csv_file, image_dir):
        self.data = pd.read_csv(csv_file)
        self.image_dir = image_dir

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_path = os.path.join(self.image_dir, f"{row['id_code']}.png")
        image = Image.open(image_path).convert("RGB")
        label = int(row["diagnosis"])
        return transform(image), label

train_set = RetinaDataset(TRAIN_CSV, TRAIN_DIR)
val_set = RetinaDataset(VAL_CSV, VAL_DIR)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)
print("Train images:", len(train_set))
print("Validation images:", len(val_set))


In [ ]:
model = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 5)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

def make_gradcam(model, image_path, class_id):
    activations, gradients = [], []

    def forward_hook(_, __, output):
        activations.append(output.detach())

    def backward_hook(_, __, output):
        gradients.append(output[0].detach())

    hook1 = model.features[-1].register_forward_hook(forward_hook)
    hook2 = model.features[-1].register_full_backward_hook(backward_hook)
    image = Image.open(image_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)
    output = model(input_tensor)
    model.zero_grad()
    output[0, class_id].backward()
    hook1.remove()
    hook2.remove()
    weights = gradients[0].mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((weights * activations[0]).sum(dim=1)).squeeze()
    cam = cam / (cam.max() + 1e-8)
    cam = cv2.resize(cam.cpu().numpy(), (224, 224))
    rgb = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, (224, 224))
    heatmap = cv2.applyColorMap((cam * 255).astype("uint8"), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    return Image.fromarray(cv2.addWeighted(rgb, 0.6, heatmap, 0.4, 0))


In [ ]:
best_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    val_acc = evaluate(model, val_loader)
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)
        print("Saved best model")

print("Best model path:", MODEL_PATH)


In [ ]:
grade_names = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

sample_id = val_set.data.iloc[0]["id_code"]
sample_path = os.path.join(VAL_DIR, f"{sample_id}.png")
sample_image = Image.open(sample_path).convert("RGB")
sample_image.resize((224, 224))


In [ ]:
input_tensor = transform(sample_image).unsqueeze(0).to(device)
with torch.no_grad():
    probs = torch.softmax(model(input_tensor), dim=1)[0]
pred_grade = int(probs.argmax().item())
confidence = float(probs[pred_grade].item())
print("Predicted grade:", pred_grade, "-", grade_names[pred_grade])
print("Confidence:", f"{confidence:.2%}")

cam_image = make_gradcam(model, sample_path, pred_grade)
cam_image


After training finishes, download `/kaggle/working/best_model.pth` and place it next to `app.py` for the Streamlit app.
